In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer

# Text Tokenization
- Word-level: Simple, high OOV (out of vocabulary) and large vocab
- Character-level: No OOV but very long sequence
- Subword: practical, balances

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
input = "Hello, how are you?"
tokens = tokenizer(input, return_tensors="pt")
print(tokens)


/Users/pengu/Developer/Deep Learning PyTorch Codebook/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'input_ids': tensor([[ 101, 7592, 1010, 2129, 2024, 2017, 1029,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


# Image/Audio/Video Tokenization
- Patch/chunk tokenization is simple & parallel
- Too small token unit -> Long n (costly attention)
- WARNING: preprocessing need to match train & inference

In [ ]:
# Image Patch Tokenizer (ViT style)
class PatchTokenizer(nn.Module):
    def __init__(self, patch_size=16, in_channels=3, embed_dim=768):
        super(PatchTokenizer, self).__init__()
        self.conv = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.conv(x)  # (B, embed_dim, H/patch_size, W/patch_size)
        x = x.flatten(2)  # (B, embed_dim, N_patches)
        x = x.transpose(1, 2)  # (B, N_patches, embed_dim)
        return x

In [ ]:
# Audio Frame Tokenizer
# frontend: waveform -> frames features (log-mel spectrogram, MFCC, etc.)
class AudioTokenizer(nn.Module):
    def __init__(self, in_features=128, embed_dim=768):
        super(AudioTokenizer, self).__init__()
        self.linear = nn.Linear(in_features, embed_dim)
    
    def forward(self, x):
        x = self.linear(x)  # (B, T, embed_dim)
        return x

In [ ]:
# Video Tubelet Tokenizer
# tubelet: 3D patch (T, H, W)
class TubeletTokenizer(nn.Module):
    def __init__(self, tubelet_size=(2, 16, 16), in_channels=3, embed_dim=768):
        super(TubeletTokenizer, self).__init__()
        self.conv = nn.Conv3d(in_channels, embed_dim, kernel_size=tubelet_size, stride=tubelet_size)
    
    def forward(self, x):
        x = self.conv(x)  # (B, embed_dim, T/tubelet_T, H/tubelet_H, W/tubelet_W)
        x = x.flatten(2)  # (B, embed_dim, N_tubelets)
        x = x.transpose(1, 2)  # (B, N_tubelets, embed_dim)
        return x

# Token vs Positional Embedding
- Token embedding: learned lookup, pretrained/fronzen/finetuned, tied input/output embedding
- Positional embedding: sinusoidal/learned table, bias (token distance), rotary (RoPE) (rotate Q/K by position)
- Input: E_token + E_pos = X_0

In [ ]:
# Learned/Tied Token Embedding
tok_embed = nn.Embedding(num_embeddings=10000, embedding_dim=768)
lm_head = nn.Linear(768, 10000)
# Tying weights
lm_head.weight = tok_embed.weight
# Pad ID fixed, mask padded tokens in loss/attention
# Rebuild entire embedding matrix if vocab changes, no OOV token, etc.

# Positional Encoding
- Give order awareness (attention gives global interaction)
- Permutation inequivariant
- Shape:
+ Token: `B x n x d` (n: vocab size, d: dimension)
+ Position: `1 x n x d`
## Absolute Positional Encoding
### Sinusoidal Positional Encoding
- Fixed: `PE(pos, 2i) = sin(pos/10000^(2i/d)), PE(pos, 2i + 1) = cos(pos/10000^(2i/d))`
- Same table can be generated, for any sequence length
- Different dimensions encode different frequencies
- Use multi-frequency periodic signals
### Learned absolute position embedding
- Use a trainable lookup table: `n_max x d`
- `X_0 = E_token + E_pos[0:n]`